# PyTorch Dataset and DataLoader from First Principles

This notebook shows how raw CSV data becomes PyTorch batches in a clear, modular sequence.

## Pipeline overview

1. import the required libraries
2. load the CSV file into pandas
3. separate features and labels
4. encode categorical features
5. convert the prepared data into tensors
6. wrap tensors in a custom dataset
7. create DataLoaders to build batches

There is no model training here. The goal is to understand loading, preparation, and batching step by step.

In [1]:
# Import the libraries used in this notebook.
import pandas as pd
import torch

# Keep the dataset path in one variable so it is easy to reuse.
csv_path = "project/dataset/diabetes_prediction_dataset.csv"

print("CSV path:", csv_path)

CSV path: project/dataset/diabetes_prediction_dataset.csv


### Load the CSV File

The first practical step is reading the CSV into pandas so we can inspect and clean the data before converting it to tensors.

In [2]:
# Load the CSV file into a pandas DataFrame.
# A DataFrame stores the raw data as a table of rows and columns.
diabetes_df = pd.read_csv(csv_path)

In [3]:
# Inspect the loaded table.
print("First 5 rows:")
display(diabetes_df.head())

print("Shape:", diabetes_df.shape)
print("Columns:", list(diabetes_df.columns))
print("\nTarget counts:")
print(diabetes_df["diabetes"].value_counts())

First 5 rows:


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


Shape: (100000, 9)
Columns: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']

Target counts:
diabetes
0    91500
1     8500
Name: count, dtype: int64


## Step 2: Prepare Features and Labels

A model learns from two pieces:

- **features**: the input columns
- **labels**: the answer column we want to predict

In this dataset, `diabetes` is the label. All remaining columns are features.

In [4]:
# Separate the target column from the feature columns.
X_df = diabetes_df.drop(columns=["diabetes"])
y_series = diabetes_df["diabetes"]

print("Feature table shape:", X_df.shape)
print("Label shape:", y_series.shape)
print("\nFeature columns:")
print(list(X_df.columns))

Feature table shape: (100000, 8)
Label shape: (100000,)

Feature columns:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']


### Encode Categorical Features

PyTorch needs numeric inputs. If a column contains text categories such as gender or smoking history, we must encode it into numbers first.

`pd.get_dummies(...)` creates binary columns that represent each category.

In [5]:
# Convert categorical columns into numeric dummy columns.
# Casting to float32 keeps the data ready for PyTorch tensor conversion.
X_encoded = pd.get_dummies(X_df, drop_first=False).astype("float32")

print("Encoded feature shape:", X_encoded.shape)
print("\nFirst 10 encoded columns:")
print(list(X_encoded.columns[:10]))

Encoded feature shape: (100000, 15)

First 10 encoded columns:
['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'gender_Female', 'gender_Male', 'gender_Other', 'smoking_history_No Info']


## Step 3: Convert the Prepared Table into Tensors

PyTorch models work with tensors, not pandas objects.

This step converts:

- the encoded feature table into a feature tensor
- the label column into a label tensor

In [6]:
# Convert the encoded table into PyTorch tensors.
# Features stay as float32, and labels are reshaped into a column vector.
X_tensor = torch.tensor(X_encoded.to_numpy(), dtype=torch.float32)
y_tensor = torch.tensor(y_series.to_numpy(), dtype=torch.float32).view(-1, 1)

print("Feature tensor shape:", X_tensor.shape)
print("Label tensor shape:", y_tensor.shape)
print("\nFirst processed feature row:")
print(X_tensor[0])
print("\nFirst processed label:")
print(y_tensor[0])

Feature tensor shape: torch.Size([100000, 15])
Label tensor shape: torch.Size([100000, 1])

First processed feature row:
tensor([ 80.0000,   0.0000,   1.0000,  25.1900,   6.6000, 140.0000,   1.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   1.0000,
          0.0000])

First processed label:
tensor([0.])


## Step 4: Define a Custom Dataset

A PyTorch `Dataset` is a lightweight wrapper around the data.

From first principles, it needs two methods:

- `__len__` to report how many examples exist
- `__getitem__` to return one feature-label pair

In [7]:
from torch.utils.data import DataLoader, Dataset


class CustomDataset(Dataset):
    def __init__(self, features, labels):
        # Store the prepared feature and label tensors.
        self.features = features
        self.labels = labels

    def __len__(self):
        # Return the number of examples in the dataset.
        return len(self.features)

    def __getitem__(self, index):
        # Return one feature row and its matching label.
        return self.features[index], self.labels[index]

## Step 5: Split the Tensor Data

The tensors now contain all examples. We still need to separate them into:

- a training split
- a test split

After splitting, each part will be wrapped in the dataset class.

In [8]:
# Create an 80/20 split using row position.
split_index = int(0.8 * len(X_tensor))

# Slice the feature and label tensors into training and test parts.
X_train = X_tensor[:split_index]
y_train = y_tensor[:split_index]
X_test = X_tensor[split_index:]
y_test = y_tensor[split_index:]

# Wrap each split in the custom dataset class.
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

print("Split index:", split_index)
print("Training tensor shape:", X_train.shape)
print("Test tensor shape:", X_test.shape)

Split index: 80000
Training tensor shape: torch.Size([80000, 15])
Test tensor shape: torch.Size([20000, 15])


## Step 6: Create DataLoaders and Inspect a Batch

A `DataLoader` sits on top of a dataset and groups samples into mini-batches.

In this final step, we will:

- create a training loader
- create a test loader
- inspect one batch to confirm the shapes are correct

In [9]:
# Create DataLoaders for the training and test datasets.
# The training loader shuffles examples, while the test loader keeps order fixed.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Read one batch so we can verify the final batching step.
train_batch_features, train_batch_labels = next(iter(train_loader))

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))
print("\nOne training batch feature shape:", train_batch_features.shape)
print("One training batch label shape:", train_batch_labels.shape)
print("\nFirst 5 labels from one training batch:")
print(train_batch_labels[:5].squeeze())

Training examples: 80000
Test examples: 20000

One training batch feature shape: torch.Size([32, 15])
One training batch label shape: torch.Size([32, 1])

First 5 labels from one training batch:
tensor([0., 0., 1., 0., 0.])
